# 30_02 — Train CustomCNN v2 From Scratch

## Goal
Train the Phase 3 advanced scratch CNN (`CustomCNN v2`) on the fixed dataset split `split_v1` using the shared Phase 1 transformation pipeline.

This notebook is designed to be:

- **Run-All safe**
- **portable across devices**
- **re-runnable** without manual toggles
- **consistent** with prior project pathing conventions
- **artifact-complete**, producing:
  - `checkpoint.pt`
  - `exported.onnx` (if export succeeds)
  - `config.json`
  - `metrics.json`
  - `loss_curve.png`
  - `accuracy_curve.png`

## Fixed Contracts
- dataset split: `split_v1`
- train/eval transforms: `configs/transforms_v1.yaml`
- model family output root: `models/cnn_scratch/customcnn_v2/`
- MLflow experiment: `AnimalClassification`

## Notes
This notebook trains a fresh model and creates a **new timestamped run directory** every time it is executed successfully.

In [ ]:
# Cell 2 — Imports

import os
import sys
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import mlflow

In [ ]:
# Cell 3 — Project root + paths (preserve prior notebook conventions)

NOTEBOOK_DIR = Path.cwd().expanduser().resolve()

def find_project_root(start: Path) -> Path:
    markers_all = ["src", "configs"]
    markers_any = [".git", "pyproject.toml", "requirements.txt"]

    for p in [start] + list(start.parents):
        has_all = all((p / m).exists() for m in markers_all)
        has_any = any((p / m).exists() for m in markers_any)
        if has_all and has_any:
            return p

    for p in [start] + list(start.parents):
        if all((p / m).exists() for m in markers_all):
            return p

    raise RuntimeError(f"Could not locate project root from: {start}")

PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)

CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"
TRACKING_DIR = PROJECT_ROOT / "mlruns"

SPLIT_ID = "split_v1"
SPLITS_DIR = DATA_DIR / "splits" / SPLIT_ID
TRAIN_CSV = SPLITS_DIR / "train.csv"
VAL_CSV = SPLITS_DIR / "val.csv"
TEST_CSV = SPLITS_DIR / "test.csv"
CLASSES_JSON = SPLITS_DIR / "classes.json"

TRANSFORMS_CONFIG_PATH = CONFIGS_DIR / "transforms_v1.yaml"
TRANSFORM_ID_TRAIN = "transforms_v1_train_runtime_aug"
TRANSFORM_ID_EVAL = "transforms_v1_eval_resize256_centercrop224_imagenetnorm"

MODEL_NAME = "customcnn_v2"
MODEL_FAMILY_DIR = MODELS_DIR / "cnn_scratch" / MODEL_NAME
REPORTS_METRICS_DIR = REPORTS_DIR / "metrics"
REPORTS_FIGURES_DIR = REPORTS_DIR / "figures"

MODEL_FAMILY_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_METRICS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TRACKING_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT     :", PROJECT_ROOT)
print("MODEL_FAMILY_DIR :", MODEL_FAMILY_DIR)
print("TRACKING_DIR     :", TRACKING_DIR)

In [ ]:
# Cell 4 — Add PROJECT_ROOT to sys.path and import project modules

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.transforms import load_transforms_config, get_train_transforms, get_eval_transforms
from src.data.dataset_loader import ImageDataset
from src.models.cnn_scratch.models import build_model
from src.models.cnn_scratch.utils import (
    atomic_save_json,
    benchmark_inference,
    build_metrics_payload,
    build_training_config,
    count_parameters,
    evaluate_model,
    export_model_to_onnx,
    fit_model,
    make_run_dir,
    model_size_mb_from_state_dict,
    restore_best_weights,
    save_checkpoint_atomic,
    save_training_curves,
)

In [ ]:
# Cell 5 — Determinism helpers

SEED = 42

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

print("[OK] Seeds initialized:", SEED)

In [ ]:
# Cell 6 — Required file checks

required_paths = {
    "train_csv": TRAIN_CSV,
    "val_csv": VAL_CSV,
    "test_csv": TEST_CSV,
    "classes_json": CLASSES_JSON,
    "transforms_yaml": TRANSFORMS_CONFIG_PATH,
}

missing_required = [name for name, path in required_paths.items() if not path.exists()]
if missing_required:
    details = "\n".join(f"- {name}: {required_paths[name]}" for name in missing_required)
    raise FileNotFoundError(f"Missing required files:\n{details}")

print("[OK] Required files exist.")
for name, path in required_paths.items():
    print(f"  - {name}: {path}")

In [ ]:
# Cell 7 — MLflow setup

mlflow.set_tracking_uri(TRACKING_DIR.as_uri())
mlflow.set_experiment("AnimalClassification")

print("MLflow tracking URI:", TRACKING_DIR.as_uri())
print("MLflow experiment  : AnimalClassification")

In [ ]:
# Cell 8 — Load split manifests and class mapping

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

with open(CLASSES_JSON, "r", encoding="utf-8") as f:
    class_to_idx = json.load(f)

idx_to_class = {v: k for k, v in class_to_idx.items()}
NUM_CLASSES = len(class_to_idx)

print("train:", len(train_df), "val:", len(val_df), "test:", len(test_df))
print("class_to_idx:", class_to_idx)

In [ ]:
# Cell 9 — Optional split preview and raw CSV sanity summary

def summarize_split(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    counts = df["label"].value_counts().sort_index()
    return pd.DataFrame({
        "split": split_name,
        "class": counts.index,
        "count": counts.values,
        "ratio": counts.values / counts.values.sum(),
    })

summary_df = pd.concat([
    summarize_split(train_df, "train"),
    summarize_split(val_df, "val"),
    summarize_split(test_df, "test"),
], ignore_index=True)

print(summary_df)

In [ ]:
# Cell 10 — Load transforms

tfm_cfg = load_transforms_config(str(TRANSFORMS_CONFIG_PATH))
train_tfm = get_train_transforms(tfm_cfg)
eval_tfm = get_eval_transforms(tfm_cfg)

print("[OK] Transforms loaded from:", TRANSFORMS_CONFIG_PATH)
print("TRAIN TRANSFORM ID:", TRANSFORM_ID_TRAIN)
print("EVAL TRANSFORM ID :", TRANSFORM_ID_EVAL)

In [ ]:
# Cell 11 — Dataset construction using shared project loader

train_ds = ImageDataset(
    split_csv=TRAIN_CSV,
    transform=train_tfm,
    classes_to_idx=class_to_idx,
    project_root=PROJECT_ROOT,
    normalize_paths=True,
    drop_missing=True,
)

val_ds = ImageDataset(
    split_csv=VAL_CSV,
    transform=eval_tfm,
    classes_to_idx=class_to_idx,
    project_root=PROJECT_ROOT,
    normalize_paths=True,
    drop_missing=True,
)

test_ds = ImageDataset(
    split_csv=TEST_CSV,
    transform=eval_tfm,
    classes_to_idx=class_to_idx,
    project_root=PROJECT_ROOT,
    normalize_paths=True,
    drop_missing=True,
)

print("[OK] Datasets built using src.data.dataset_loader.ImageDataset.")
print("train:", len(train_ds), "val:", len(val_ds), "test:", len(test_ds))

In [ ]:
# Cell 12 — Device and DataLoaders

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = min(8, os.cpu_count() or 2)
PIN_MEMORY = True if DEVICE == "cuda" else False

BATCH_SIZE = 64 if DEVICE == "cuda" else 16

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print("DEVICE      :", DEVICE)
print("BATCH_SIZE  :", BATCH_SIZE)
print("NUM_WORKERS :", NUM_WORKERS)
print("PIN_MEMORY  :", PIN_MEMORY)

In [ ]:
# Cell 13 — Batch sanity check

xb, yb = next(iter(train_loader))

assert xb.ndim == 4 and xb.shape[1:] == (3, 224, 224), f"Unexpected batch shape: {tuple(xb.shape)}"
assert yb.ndim == 1, f"Unexpected labels shape: {tuple(yb.shape)}"
assert torch.isfinite(xb).all(), "Non-finite values in training batch."

print("[OK] Batch sanity check passed:", tuple(xb.shape), tuple(yb.shape))
print("Unique labels in batch:", sorted(torch.unique(yb).tolist()))

In [ ]:
# Cell 14 — Hyperparameters and training config

EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT_P = 0.5
GRAD_CLIP_MAX_NORM = 1.0

training_params = {
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "dropout_p": DROPOUT_P,
    "grad_clip_max_norm": GRAD_CLIP_MAX_NORM,
    "optimizer": "Adam",
    "scheduler": {
        "name": "ReduceLROnPlateau",
        "mode": "min",
        "factor": 0.3,
        "patience": 3,
    },
    "loss_fn": "CrossEntropyLoss",
    "seed": SEED,
}

config_payload = build_training_config(
    model_name=MODEL_NAME,
    split_id=SPLIT_ID,
    transform_id_train=TRANSFORM_ID_TRAIN,
    transform_id_eval=TRANSFORM_ID_EVAL,
    dataset_version="prepared_dataset_current",
    training_params=training_params,
    extra={
        "num_classes": NUM_CLASSES,
        "class_to_idx": class_to_idx,
    },
)

print(json.dumps(config_payload, indent=2))

In [ ]:
# Cell 15 — Create run directory and save config early

RUN_DIR = make_run_dir(MODEL_FAMILY_DIR, prefix="run")
CHECKPOINT_PATH = RUN_DIR / "checkpoint.pt"
ONNX_PATH = RUN_DIR / "exported.onnx"
CONFIG_PATH = RUN_DIR / "config.json"
METRICS_PATH = RUN_DIR / "metrics.json"

atomic_save_json(CONFIG_PATH, config_payload)

print("RUN_DIR        :", RUN_DIR)
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)
print("ONNX_PATH      :", ONNX_PATH)
print("CONFIG_PATH    :", CONFIG_PATH)
print("METRICS_PATH   :", METRICS_PATH)

In [ ]:
# Cell 16 — Build model, optimizer, scheduler, criterion

model = build_model(
    model_name=MODEL_NAME,
    num_classes=NUM_CLASSES,
    input_channels=3,
    dropout_p=DROPOUT_P,
).to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.3,
    patience=3,
)

param_count = count_parameters(model)
size_mb = model_size_mb_from_state_dict(model)

print(model)
print("Parameter count:", param_count)
print("Approx model size (MB):", round(size_mb, 3))

In [ ]:
# Cell 17 — Train model

history, best_state = fit_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=DEVICE,
    num_classes=NUM_CLASSES,
    epochs=EPOCHS,
    grad_clip_max_norm=GRAD_CLIP_MAX_NORM,
)

print("[OK] Training complete.")
print("Best epoch:", best_state.get("epoch"))
print("Best val macro F1:", best_state.get("best_val_macro_f1"))

In [ ]:
# Cell 18 — Restore best weights and save checkpoint

model = restore_best_weights(model, best_state)

checkpoint_payload = {
    "model_name": MODEL_NAME,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": best_state.get("optimizer_state_dict"),
    "best_epoch": best_state.get("epoch"),
    "best_val_macro_f1": best_state.get("best_val_macro_f1"),
    "best_val_loss": best_state.get("best_val_loss"),
    "best_val_accuracy": best_state.get("best_val_accuracy"),
    "class_to_idx": class_to_idx,
    "config": config_payload,
}

save_checkpoint_atomic(CHECKPOINT_PATH, checkpoint_payload)

print("[OK] Checkpoint saved:", CHECKPOINT_PATH)

In [ ]:
# Cell 19 — Evaluate best model on test set

test_metrics = evaluate_model(
    model=model,
    loader=test_loader,
    criterion=criterion,
    device=DEVICE,
    num_classes=NUM_CLASSES,
)

print("Test loss     :", round(test_metrics["loss"], 4))
print("Test accuracy :", round(test_metrics["accuracy"], 4))
print("Test macro F1 :", round(test_metrics["macro_f1"], 4))

In [ ]:
# Cell 20 — Benchmark inference cost on test loader

benchmark_metrics = benchmark_inference(
    model=model,
    loader=test_loader,
    device=DEVICE,
    warmup_batches=5,
    timed_batches=20,
)

print(json.dumps(benchmark_metrics, indent=2))

In [ ]:
# Cell 21 — Save training curves

curve_paths = save_training_curves(history, RUN_DIR)

print("Saved curves:")
for name, path in curve_paths.items():
    print(f"  - {name}: {path}")

In [ ]:
# Cell 22 — Export ONNX safely (non-fatal if export support is unavailable)

onnx_export_status = {
    "attempted": True,
    "succeeded": False,
    "path": str(ONNX_PATH),
    "error": None,
}

try:
    export_model_to_onnx(
        model=model,
        export_path=ONNX_PATH,
        input_shape=(1, 3, 224, 224),
        device=DEVICE,
        opset_version=17,
    )
    onnx_export_status["succeeded"] = True
    print("[OK] ONNX exported:", ONNX_PATH)

except Exception as e:
    onnx_export_status["error"] = f"{type(e).__name__}: {e}"
    print("[WARNING] ONNX export failed. Training artifacts remain valid.")
    print("ONNX export error:", onnx_export_status["error"])

In [ ]:
# Cell 23 — Build and save metrics payload

metrics_payload = build_metrics_payload(
    history=history,
    best_state=best_state,
    test_metrics=test_metrics,
    benchmark_metrics=benchmark_metrics,
    parameter_count=param_count,
    model_size_mb=size_mb,
)

metrics_payload["onnx_export"] = onnx_export_status

atomic_save_json(METRICS_PATH, metrics_payload)

print("[OK] Metrics saved:", METRICS_PATH)

In [ ]:
# Cell 24 — Save lightweight report copies to reports/metrics

report_metrics_path = REPORTS_METRICS_DIR / f"{MODEL_NAME}_{RUN_DIR.name}_metrics.json"
atomic_save_json(report_metrics_path, metrics_payload)

print("[OK] Report metrics copy saved:", report_metrics_path)

In [ ]:
# Cell 25 — MLflow logging

with mlflow.start_run(run_name=f"{MODEL_NAME}_{RUN_DIR.name}"):
    mlflow.log_param("stage", "cnn_scratch_training")
    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_param("split_id", SPLIT_ID)
    mlflow.log_param("transform_id_train", TRANSFORM_ID_TRAIN)
    mlflow.log_param("transform_id_eval", TRANSFORM_ID_EVAL)
    mlflow.log_param("device", DEVICE)
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("learning_rate", LEARNING_RATE)
    mlflow.log_param("weight_decay", WEIGHT_DECAY)
    mlflow.log_param("dropout_p", DROPOUT_P)
    mlflow.log_param("grad_clip_max_norm", GRAD_CLIP_MAX_NORM)
    mlflow.log_param("num_workers", NUM_WORKERS)
    mlflow.log_param("parameter_count", param_count)
    mlflow.log_param("model_size_mb", size_mb)
    mlflow.log_param("onnx_export_succeeded", onnx_export_status["succeeded"])

    mlflow.log_metric("best_epoch", float(best_state.get("epoch", -1)))
    mlflow.log_metric("best_val_macro_f1", float(best_state.get("best_val_macro_f1", float("nan"))))
    mlflow.log_metric("best_val_loss", float(best_state.get("best_val_loss", float("nan"))))
    mlflow.log_metric("best_val_accuracy", float(best_state.get("best_val_accuracy", float("nan"))))

    mlflow.log_metric("test_loss", float(test_metrics["loss"]))
    mlflow.log_metric("test_accuracy", float(test_metrics["accuracy"]))
    mlflow.log_metric("test_macro_f1", float(test_metrics["macro_f1"]))

    mlflow.log_metric("latency_ms_per_batch", float(benchmark_metrics["latency_ms_per_batch"]))
    mlflow.log_metric("latency_ms_per_image", float(benchmark_metrics["latency_ms_per_image"]))
    mlflow.log_metric("throughput_img_per_sec", float(benchmark_metrics["throughput_img_per_sec"]))

    mlflow.log_artifact(str(CONFIG_PATH))
    mlflow.log_artifact(str(METRICS_PATH))
    mlflow.log_artifact(str(CHECKPOINT_PATH))
    mlflow.log_artifact(str(curve_paths["loss_curve"]))
    mlflow.log_artifact(str(curve_paths["accuracy_curve"]))

    if onnx_export_status["succeeded"] and ONNX_PATH.exists():
        mlflow.log_artifact(str(ONNX_PATH))

print("[OK] MLflow logging complete.")

## Result

This notebook completed the full Phase 3 training run for `CustomCNN v2` and produced:

- checkpoint
- ONNX export (if successful)
- run config
- metrics
- training curves
- MLflow logs
- inference benchmarking metrics

This notebook mirrors the Phase 3 `CustomCNN v1` training notebook to preserve fair benchmarking and structural consistency.